In [3]:
import torch  # deep learning library for tensors
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [8]:
with open('wizard_of_oz.txt','r', encoding='utf-8') as f:
    text = f.read()
print(len(text))
for line in text.split('\n'):# .split('\n') is a Python string method that breaks a string into a list of substrings wherever it finds the specified separator. 
#"hello\nworld\nfoo".split('\n')
# Result: ['hello', 'world', 'foo']
    if "chapter 1" in line.lower():
        break
    print(line)

232314
﻿DOROTHY AND THE WIZARD IN OZ

  BY

  L. FRANK BAUM

  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ, OZMA OF OZ, ETC.

  ILLUSTRATED BY JOHN R. NEILL

  BOOKS OF WONDER WILLIAM MORROW & CO., INC. NEW YORK


  [Illustration]


  COPYRIGHT 1908 BY L. FRANK BAUM

  ALL RIGHTS RESERVED


         *       *       *       *       *


  [Illustration]


  DEDICATED TO HARRIET A. B. NEAL.


         *       *       *       *       *


To My Readers


It's no use; no use at all. The children won't let me stop telling tales
of the Land of Oz. I know lots of other stories, and I hope to tell
them, some time or another; but just now my loving tyrants won't allow
me. They cry: "Oz--Oz! more about Oz, Mr. Baum!" and what can I do but
obey their commands?

This is Our Book--mine and the children's. For they have flooded me with
thousands of suggestions in regard to it, and I have honestly tried to
adopt as many of these suggestions as could be fitted into one story.

After the wonderful success

In [9]:
chars = sorted(set(text))  # get all unique characters in the text, sorted alphabetically
print(chars)               # print the full character vocabulary
print(len(chars))          # print vocabulary size (number of unique characters)

['\n', ' ', '!', '"', '&', "'", '(', ')', '*', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\ufeff']
81


In [10]:
my_list = [1,2,3,4,5,6,7]
print(list(enumerate(my_list)))  # wrap in list() to see the (index, value) pairs

[(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7)]


In [11]:
# --- enumerate() examples ---

fruits = ['apple', 'banana', 'cherry']

# Without list() - just prints the object, not useful
print(enumerate(fruits))          # <enumerate object at 0x...>

# With list() - forces it to show all pairs
print(list(enumerate(fruits)))    # [(0, 'apple'), (1, 'banana'), (2, 'cherry')]

# Common use: loop with index and value at the same time
for i, fruit in enumerate(fruits):
    print(i, fruit)
# 0 apple
# 1 banana
# 2 cherry

# You can also start the index at a different number
print(list(enumerate(fruits, start=1)))  # [(1, 'apple'), (2, 'banana'), (3, 'cherry')]

[(0, 'apple'), (1, 'banana'), (2, 'cherry')]
0 apple
1 banana
2 cherry
[(1, 'apple'), (2, 'banana'), (3, 'cherry')]


In [12]:
string_to_int = { ch:i for i,ch in enumerate(chars) }  # maps each character to a unique integer
int_to_string = { i:ch for i,ch in enumerate(chars) }  # maps each integer back to its character
encode = lambda string: [string_to_int[char] for char in string]       # converts a string into a list of integers
decode = lambda Integer: ''.join([int_to_string[int] for int in Integer])  # converts a list of integers back into a string
#'hello'  →  encode  →  [34, 21, 38, 38, 41]  →  decode  →  'hello'

encoded_hello = encode('hello')   # encode the word 'hello' into a list of integers
decoded_hello = decode(encoded_hello)  # decode those integers back into a string
print(decoded_hello)  # should print 'hello'


hello


# rewrite all the functions and vars

In [13]:
data = torch.tensor(encode(text), dtype=torch.long)  # encode entire text to integers, wrap in tensor (dtype=long required for embeddings)
print(data[:100])  # print first 100 values to verify encoding

tensor([80, 28, 39, 42, 39, 44, 32, 49,  1, 25, 38, 28,  1, 44, 32, 29,  1, 47,
        33, 50, 25, 42, 28,  1, 33, 38,  1, 39, 50,  0,  0,  1,  1, 26, 49,  0,
         0,  1,  1, 36, 11,  1, 30, 42, 25, 38, 35,  1, 26, 25, 45, 37,  0,  0,
         1,  1, 25, 45, 44, 32, 39, 42,  1, 39, 30,  1, 44, 32, 29,  1, 47, 33,
        50, 25, 42, 28,  1, 39, 30,  1, 39, 50,  9,  1, 44, 32, 29,  1, 36, 25,
        38, 28,  1, 39, 30,  1, 39, 50,  9,  1])


In [14]:
n = int(0.8 * len(data))  # index at 80% of total data — the split point
train_data = data[:n]     # first 80% for training
val_data = data[n:]       # last 20% for validation (never seen during training)

In [15]:
block_size = 8  # context window — max number of characters the model looks at at once

x = train_data[:block_size]    # input: characters 0–7
y = train_data[1:block_size+1] # target: characters 1–8 (shifted by 1, so each target is the next char after the input)

for t in range(block_size):         # loop through each position in the block
    context = x[:t+1]               # context grows: 1 char on first step, 2 on second, up to block_size
    target = y[t]                   # the correct next character to predict at this step
    print('when input is', context, 'target is', target)  # show all 8 training examples inside this block

when input is tensor([80]) target is tensor(28)
when input is tensor([80, 28]) target is tensor(39)
when input is tensor([80, 28, 39]) target is tensor(42)
when input is tensor([80, 28, 39, 42]) target is tensor(39)
when input is tensor([80, 28, 39, 42, 39]) target is tensor(44)
when input is tensor([80, 28, 39, 42, 39, 44]) target is tensor(32)
when input is tensor([80, 28, 39, 42, 39, 44, 32]) target is tensor(49)
when input is tensor([80, 28, 39, 42, 39, 44, 32, 49]) target is tensor(1)
